<a href="https://colab.research.google.com/github/offiong117/VideoGameDesignPortfolio/blob/main/E8_GraphRAG_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise E8 – Colab run (import + Coding Tasks 1 and 2)
Dr. Babaie's import code, unchanged, plus completed Coding Task 1 and Task 2 cells.

**Before Run all**, upload two files with the folder icon on the left:
1. `CMiO_first500_rows.csv` (the first 500 rows of the class dataset, which is all the notebook imports)
2. your own `Neo4j-...-Created-....txt` credentials file from AuraDB

The password is read from that file and never printed.

In [1]:
%pip install -q -U neo4j python-dotenv

In [2]:
# Imports and configuration
import os, json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

PROJECT_DIR = Path.cwd()
CSV_FILE = PROJECT_DIR / "CMiO_first500_rows.csv"
DATASET_ID = "E8_CMIO_REAL"
IMPORT_LIMIT = 500
if not CSV_FILE.is_file():
    raise FileNotFoundError("Upload CMiO_first500_rows.csv with the folder icon on the left, then run again.")
df = pd.read_csv(CSV_FILE, dtype=str, low_memory=False)
print("Rows:", len(df), "| Columns:", len(df.columns))

def choose_column(frame, *candidates):
    lookup = {str(c).strip().upper(): c for c in frame.columns}
    for candidate in candidates:
        found = lookup.get(candidate.upper())
        if found is not None:
            return found
    return None

column_map = {
    "deposit_id": choose_column(df, "DEPOSIT_UID", "DEPOSIT_ID"),
    "deposit_name": choose_column(df, "DEPOSIT_NAME"),
    "deposit_type": choose_column(df, "DEPOSIT_TYPE"),
    "deposit_group": choose_column(df, "DEPOSIT_GROUP"),
    "deposit_environment": choose_column(df, "DEPOSIT_ENVIRONMENT"),
    "sample_id": choose_column(df, "SAMPLE_UID"),
    "analysis_id": choose_column(df, "DEPOSIT_ANALYSIS_ID"),
}
column_map

Rows: 500 | Columns: 7


{'deposit_id': 'DEPOSIT_UID',
 'deposit_name': 'DEPOSIT_NAME',
 'deposit_type': 'DEPOSIT_TYPE',
 'deposit_group': 'DEPOSIT_GROUP',
 'deposit_environment': 'DEPOSIT_ENVIRONMENT',
 'sample_id': 'SAMPLE_UID',
 'analysis_id': 'DEPOSIT_ANALYSIS_ID'}

In [3]:
# Neo4j driver setup — import and credentials are deliberately in this cell.
credential_files = sorted(PROJECT_DIR.glob("Neo4j-*-Created-*.txt"))
if len(credential_files) != 1:
    raise RuntimeError(
        f"Expected exactly one Aura credentials file in {PROJECT_DIR}; found {len(credential_files)}."
    )
CREDENTIALS_FILE = credential_files[0]
load_dotenv(CREDENTIALS_FILE, override=True)

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
if not URI or not PASSWORD:
    raise RuntimeError("Missing NEO4J_URI or NEO4J_PASSWORD in the credentials file.")
if not URI.startswith(("neo4j+s://", "neo4j+ssc://")):
    raise ValueError("Aura requires a secure URI such as neo4j+s://...")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print("Connected to AuraDB. Database:", DATABASE)
# Password and complete credentials are deliberately never printed.

Connected to AuraDB. Database: neo4j


In [4]:
# Cleaning functions: missing values must not become literal "nan" nodes.
def clean_value(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return None if not text or text.lower() in {"nan", "none", "null"} else text

def value_at(row, concept):
    header = column_map.get(concept)
    return clean_value(row[header]) if header is not None else None

UNKNOWN_TYPES = {"unknown", "undetermined", "unclassified", "not classified", "n/a", "na"}
source_rows = df if IMPORT_LIMIT is None else df.head(IMPORT_LIMIT)
print("CSV rows selected for import:", len(source_rows))

CSV rows selected for import: 500


In [5]:
# All writes use REAL CSV values; no illustrative entities are created.
# A scoped dataset_id allows students to query only their E8 records.
import_cypher = """
MERGE (d:Deposit {dataset_id:$dataset_id, uid:$deposit_id})
SET d.name = coalesce($deposit_name, d.name),
    d.environment = coalesce($environment, d.environment),
    d.deposit_group = coalesce($deposit_group, d.deposit_group),
    d.source_file = $source_file
FOREACH (_ IN CASE WHEN $deposit_type IS NULL THEN [] ELSE [1] END |
    MERGE (dt:DepositType {dataset_id:$dataset_id, name:$deposit_type})
    MERGE (d)-[:CLASSIFIED_AS]->(dt)
)
FOREACH (_ IN CASE WHEN $sample_id IS NULL THEN [] ELSE [1] END |
    MERGE (s:Sample {dataset_id:$dataset_id, uid:$sample_id})
    MERGE (s)-[:TAKEN_FROM]->(d)
    FOREACH (__ IN CASE WHEN $analysis_id IS NULL THEN [] ELSE [1] END |
        MERGE (a:Analysis {dataset_id:$dataset_id, uid:$analysis_id})
        MERGE (s)-[:HAS_ANALYSIS]->(a)
    )
)
"""

stats = {"source_rows": 0, "rows_with_deposit_id": 0,
         "rows_missing_deposit_id": 0, "rows_with_unknown_type": 0}
with driver.session(database=DATABASE) as session:
    for _, row in source_rows.iterrows():
        stats["source_rows"] += 1
        deposit_id = value_at(row, "deposit_id")
        if deposit_id is None:
            stats["rows_missing_deposit_id"] += 1
            continue
        deposit_type = value_at(row, "deposit_type")
        if deposit_type is None or deposit_type.casefold() in UNKNOWN_TYPES:
            stats["rows_with_unknown_type"] += 1
            deposit_type = None
        params = {
            "dataset_id": DATASET_ID,
            "deposit_id": deposit_id,
            "deposit_name": value_at(row, "deposit_name"),
            "environment": value_at(row, "deposit_environment"),
            "deposit_group": value_at(row, "deposit_group"),
            "deposit_type": deposit_type,
            "sample_id": value_at(row, "sample_id"),
            "analysis_id": value_at(row, "analysis_id"),
            "source_file": CSV_FILE.name,
        }
        session.run(import_cypher, **params).consume()  # Confirm each write completed.
        stats["rows_with_deposit_id"] += 1

print("Import summary:", stats)
if stats["rows_with_deposit_id"] == 0:
    raise RuntimeError("No deposits imported. Inspect deposit identifiers and CSV headers.")

Import summary: {'source_rows': 500, 'rows_with_deposit_id': 500, 'rows_missing_deposit_id': 0, 'rows_with_unknown_type': 0}


In [6]:
# Count real nodes by label; dataset_id isolates this exercise's imported records.
count_cypher = """
MATCH (n)
WHERE n.dataset_id = $dataset_id
RETURN labels(n)[0] AS label, count(n) AS nodes
ORDER BY label
"""
records, _, _ = driver.execute_query(count_cypher, dataset_id=DATASET_ID, database_=DATABASE)
display(pd.DataFrame([r.data() for r in records]))

# Inspect up to ten actual sample → deposit paths (if sample IDs exist).
path_cypher = """
MATCH (s:Sample {dataset_id:$dataset_id})-[:TAKEN_FROM]->(d:Deposit)
RETURN s.uid AS sample_id, d.uid AS deposit_id, d.name AS deposit_name
ORDER BY deposit_id, sample_id LIMIT 10
"""
records, _, _ = driver.execute_query(path_cypher, dataset_id=DATASET_ID, database_=DATABASE)
display(pd.DataFrame([r.data() for r in records]))

,label,nodes
0,Analysis,500
1,Deposit,86
2,DepositType,33
3,Sample,225


,sample_id,deposit_id,deposit_name
0,AU.7372019,AUS.NSW.335835,Cadia Hill
1,AU.7372020,AUS.NSW.335835,Cadia Hill
2,AU.7372104,AUS.NSW.335835,Cadia Hill
3,AU.1457754,AUS.NSW.335841,Woodlawn
4,AU.1457762,AUS.NSW.335841,Woodlawn
5,AU.1457778,AUS.NSW.335841,Woodlawn
6,AU.1457779,AUS.NSW.335841,Woodlawn
7,AU.1457782,AUS.NSW.335841,Woodlawn
8,AU.1457784,AUS.NSW.335841,Woodlawn
9,AU.1457785,AUS.NSW.335841,Woodlawn


## Coding Task 1 — Retrieve deposits of one recorded type
Screenshot this cell (code + output) for Question 32.

In [7]:
# Coding Task 1: deposits whose recorded type contains "pegmatite"
from neo4j import GraphDatabase

# 1. Reconnect (URI, USER, PASSWORD were loaded earlier; the password is not printed)
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()

# 2-3. Parameterized Cypher, limited to this dataset, sorted by deposit ID
task1_q = """
MATCH (d:Deposit {dataset_id:$dataset_id})-[:CLASSIFIED_AS]->(t:DepositType)
WHERE toLower(t.name) CONTAINS $keyword
RETURN d.uid AS deposit_id, d.name AS deposit_name, t.name AS deposit_type
ORDER BY deposit_id
"""
records, _, _ = driver.execute_query(
    task1_q, dataset_id=DATASET_ID, keyword="pegmatite", database_=DATABASE
)

# 4. Show as a DataFrame and count distinct deposits
task1_df = pd.DataFrame([r.data() for r in records])
display(task1_df)
n = task1_df["deposit_id"].nunique() if not task1_df.empty else 0
print("Distinct pegmatite deposits found:", n)

,deposit_id,deposit_name,deposit_type
0,NAM.ER.653137,Rubicon,LCT pegmatite


Distinct pegmatite deposits found: 1


## Coding Task 2 — Investigate unknown classifications
Screenshot this cell (code + output) for Question 33.

In [8]:
# Coding Task 2: deposits with no CLASSIFIED_AS relationship
task2_q = """
MATCH (d:Deposit {dataset_id:$dataset_id})
WHERE NOT EXISTS { MATCH (d)-[:CLASSIFIED_AS]->(:DepositType) }
RETURN d.uid AS deposit_id, d.name AS deposit_name, d.environment AS deposit_environment
ORDER BY deposit_id LIMIT 15
"""
records, _, _ = driver.execute_query(task2_q, dataset_id=DATASET_ID, database_=DATABASE)
task2_df = pd.DataFrame([r.data() for r in records])
if task2_df.empty:
    print("No unclassified deposits to display.")
else:
    display(task2_df)

# Count across the entire imported E8 graph, not just the 15 shown
count_q = """
MATCH (d:Deposit {dataset_id:$dataset_id})
WHERE NOT EXISTS { MATCH (d)-[:CLASSIFIED_AS]->(:DepositType) }
RETURN count(d) AS unclassified_deposits
"""
records, _, _ = driver.execute_query(count_q, dataset_id=DATASET_ID, database_=DATABASE)
print("Unclassified deposits in the whole E8 graph:", records[0]["unclassified_deposits"])

No unclassified deposits to display.
Unclassified deposits in the whole E8 graph: 0


In [ ]:
driver.close()
print("Neo4j driver closed.")